In [1]:
import ProcessOptimizer
from ProcessOptimizer.space import Integer, Real, Space, Categorical
from ProcessOptimizer.doe.optimal_design import get_optimal_DOE
from ProcessOptimizer.doe.doe_transform import doe_to_real_space
import numpy as np

In [2]:
print(ProcessOptimizer.__version__)

1.0.2


In [2]:
from ProcessOptimizer.doe.doe_utils import generate_replicas_and_sort

design_points = np.array([[1, 2], [3, 4]])
result = generate_replicas_and_sort(
    design_points, 2, "random_but_group_replicates"
    )

In [3]:
print(result)

[[3 4]
 [3 4]
 [1 2]
 [1 2]]


In [11]:
assert result[0].all() == result[2].all()

In [13]:
result[2][0]

1

In [8]:
result[2]

array([1, 2], dtype=object)

In [9]:
factor_space = Space(dimensions=[Integer(10, 40, name='int_var1'),
                                 Integer(-40, 520, name='int_var2'),
                                 Real(0.4, 117.7, name='real_var1'),
                                 Categorical(['A', 'B'], name='cat_var1'),
                                # Categorical(['X', 'Y'], name='cat_var2'),
                                ])

In [10]:
for factor in factor_space.dimensions:
    print(factor.name)
    if isinstance(factor, Categorical):
        print(len(factor.categories))

int_var1
int_var2
real_var1
cat_var1
2


In [11]:
design, factor_names = get_optimal_DOE(factor_space, 28, 
                                       design_type='response',
                                       model=None,
                                       replicates=1,
                                       sorting='randomized',
                                       res=31)



#print(design)

In [12]:
print(design)
print(factor_names)

[['25' '520' '59.05' 'A']
 ['40' '520' '0.4' 'A']
 ['10' '520' '0.4' 'A']
 ['40' '-40' '117.7' 'B']
 ['40' '520' '117.7' 'B']
 ['40' '-40' '0.4' 'B']
 ['10' '-40' '55.14' 'A']
 ['10' '520' '117.7' 'A']
 ['25' '-40' '59.05' 'B']
 ['24' '520' '0.4' 'B']
 ['10' '-40' '0.4' 'B']
 ['24' '-40' '0.4' 'A']
 ['40' '-40' '117.7' 'A']
 ['10' '520' '55.14' 'B']
 ['10' '-40' '117.7' 'A']
 ['23' '221' '117.7' 'B']
 ['10' '520' '117.7' 'B']
 ['40' '-40' '117.7' 'A']
 ['40' '-40' '0.4' 'A']
 ['40' '520' '117.7' 'B']
 ['10' '-40' '117.7' 'B']
 ['40' '259' '51.23' 'A']
 ['10' '259' '0.4' 'B']
 ['40' '520' '117.7' 'A']
 ['10' '221' '0.4' 'A']
 ['40' '221' '51.23' 'B']
 ['40' '520' '0.4' 'B']
 ['23' '259' '117.7' 'A']]
['int_var1', 'int_var2', 'real_var1', 'cat_var1']


In [13]:
model = '(ul_indicator+ul_indicator2+ul_indicator4+antibody)**2+pow(ul_indicator, 2)+pow(ul_indicator2, 2)+pow(ul_indicator4, 2)'

from patsy import ModelDesc
desc = ModelDesc.from_formula(model)
desc.describe()

'~ ul_indicator + ul_indicator2 + ul_indicator4 + antibody + ul_indicator:ul_indicator2 + ul_indicator:ul_indicator4 + ul_indicator:antibody + ul_indicator2:ul_indicator4 + ul_indicator2:antibody + ul_indicator4:antibody + pow(ul_indicator, 2) + pow(ul_indicator2, 2) + pow(ul_indicator4, 2)'

In [14]:
cat_var_levels = [None, None, 2]
include_powers = False

In [15]:
if include_powers is True and any(level for level in cat_var_levels):
    print("something is up")

In [21]:
for factor in factor_space.dimensions:
    if isinstance(factor, Integer):
        print(factor.name, "is integer")
    elif isinstance(factor, Real):
        print(factor.name, "is real")
    elif isinstance(factor, Categorical):
        print(factor.name, "is categorical")
        print(len(factor.categories))



int_var1 is integer
int_var2 is integer
real_var1 is real
cat_var1 is categorical
2


In [20]:

space = Space([Real(-100, 100, name='x1'), Real(0, 1, name='x2'), Categorical(['A', 'B'], name='x3')])
design = np.array([[0, 0, 0], [1, 1, 1]])
result = doe_to_real_space(design, space)
assert result[0] == [-100, 0, 'A']
assert result[1] == [100, 1, 'B']

In [21]:

def test_categorical_doe_to_real_space_rounding():
    from ProcessOptimizer.space import Categorical
    space = Space([Real(-100, 100, name='x1'), Real(0, 1, name='x2'),
                   Categorical(['A', 'B'], name='x3')])
    design = np.array([[0, 0, 0.3], [1, 1, 0.8]])
    result = doe_to_real_space(design, space)
    assert result[0] == [-100, 0, 'A']
    assert result[1] == [100, 1, 'B']



def test_categorical_doe_to_real_space_edge():
    from ProcessOptimizer.space import Categorical
    space = Space([Real(-100, 100, name='x1'), Real(0, 1, name='x2'),
                   Categorical(['A', 'B'], name='x3')])
    design = np.array([[0, 0, -42], [1, 1, 31]])
    result = doe_to_real_space(design, space)
    assert result[0] == [-100, 0, 'A']
    assert result[1] == [100, 1, 'B']

test_categorical_doe_to_real_space_edge()
test_categorical_doe_to_real_space_rounding()

In [11]:
from ProcessOptimizer.doe.doe_utils import generate_replicas_and_sort
import numpy as np

def test_generate_replicas_and_sort_random_but_group_replicates():
    design_points = np.array([[1, 2], [3, 4]])
    result = generate_replicas_and_sort(design_points, 2, "random_but_group_replicates")
    assert result.shape == (4, 2)
    assert result[1].all() == result[2].all()

out = test_generate_replicas_and_sort_random_but_group_replicates()

Went here


In [16]:
from ProcessOptimizer.doe.doe_utils import sanitize_names_for_patsy

def test_sanitize_names_for_patsy():
    factor_names = [
        "Factor 1",
        "Factor-2",
        "Factor+3",
        "Factor*4",
        "Factor/5",
        "Factor:6",
        "Factor^7",
        "Factor=8",
        "Factor~9",
        "Factor$10",
        "Factor(11)",
        "Factor[12]",
        "Factor{13}",
    ]
    expected = [
        "Factor_1",
        "Factor_2",
        "Factor_3",
        "Factor_4",
        "Factor_5",
        "Factor_6",
        "Factor_7",
        "Factor_8",
        "Factor_9",
        "Factor10",
        "Factor11",
        "Factor12",
        "Factor13",
    ]
    result = sanitize_names_for_patsy(factor_names)
    assert result == expected

out = test_sanitize_names_for_patsy()

In [22]:
space = ProcessOptimizer.space.normalize_dimensions(factor_space)
print(space)

Space([Integer(low=10, high=40),
       Integer(low=-40, high=520),
       Real(low=0.4, high=117.7, prior='uniform', transform='normalize'),
       Categorical(categories=('A', 'B'), prior=None)])


In [13]:
from ProcessOptimizer.doe.optimal_design import build_optimal_design

def optimal_design_space():
    return Space([Real(20, 100, name='x1'), Real(0, 1, name='x2'),
                  Categorical(['A', 'B'], name='x3')])


def test_build_optimal_design_with_categorical():
    factor_names = ['x1', 'x2', 'x3']
    space = optimal_design_space()

    result = build_optimal_design(factor_names, run_count=12, space=space)

    print(result)

    assert result.shape == (12, 3)
    assert np.all(np.isin(result[:, 2], [-1, 1]))

test_build_optimal_design_with_categorical()

[[ 1.          1.          1.        ]
 [ 1.          1.         -1.        ]
 [-1.          1.          1.        ]
 [-1.          1.         -1.        ]
 [ 0.4        -1.          1.        ]
 [ 0.         -1.         -1.        ]
 [-1.         -1.         -1.        ]
 [ 1.         -1.         -1.        ]
 [-1.         -1.          1.        ]
 [ 0.          0.         -1.        ]
 [ 1.         -0.2         1.        ]
 [ 0.09840003 -0.32569928 -1.        ]]


In [2]:
def test_categorical_doe_to_real_space():
    space = Space([Real(-100, 100, name='x1'), Real(0, 1, name='x2'),
                   Categorical(['A', 'B'], name='x3')])
    design = np.array([[0, 0, 0], [1, 1, 1]])
    result = doe_to_real_space(design, space)

    assert result[0] == [-100, 0, 'A']
    assert result[1] == [100, 1, 'B']
    return result



converted = test_categorical_doe_to_real_space()
print(converted)

from ProcessOptimizer.doe.doe_utils import generate_replicas_and_sort


result = generate_replicas_and_sort(converted, 2, False)
print(result)


print(result.shape)


[[np.float64(-100.0), np.float64(0.0), 'A'], [np.float64(100.0), np.float64(1.0), 'B']]
[[np.float64(-100.0) np.float64(0.0) 'A']
 [np.float64(100.0) np.float64(1.0) 'B']
 [np.float64(-100.0) np.float64(0.0) 'A']
 [np.float64(100.0) np.float64(1.0) 'B']]
(4, 3)


In [22]:
def try_convert_to_float(s):
    """
    Try to convert a string to a float. If it fails, return the string.

    :param s: The string to convert
    :type s: str

    :return: The float or the string
    :rtype: float or str
    """
    try:
        return float(s)
    except Exception:
        return s

In [ ]:
from ProcessOptimizer.doe.doe_utils import generate_replicas_and_sort

def test_generate_replicas_and_sort_no_sorting():
    design_points = np.array([[1, 2, 'A'], [3, 4, 'B']], dtype=object)
    #print(design_points)
    result = generate_replicas_and_sort(design_points, 2, False)
    print(result)
    expected = np.array([[1, 2], [3, 4], [1, 2], [3, 4]])
    #np.testing.assert_array_equal(result, expected)
    return result

result = test_generate_replicas_and_sort_no_sorting()

print(result)

[[1 2 'A']
 [3 4 'B']]
[[1 2 'A']
 [3 4 'B']
 [1 2 'A']
 [3 4 'B']]
[[1 2 'A']
 [3 4 'B']
 [1 2 'A']
 [3 4 'B']]


In [ ]:
data = np.array([['1.5', '2.7', '3.2'],
                 ['4.1', '5.3', '6.0'],
                 ['7.2', 'invalid', '9.9']])

# Function to convert strings to floats or NaN
def try_convert_to_float(s):
    try:
        return float(s)
    except ValueError:
        return np.nan

# Vectorize the conversion function
vectorized_conversion = np.vectorize(try_convert_to_float)

# Apply the conversion function to the entire array
converted_data = vectorized_conversion(data)

print(converted_data)

ValueError: could not convert string to float: 'invalid'

In [57]:
data = np.array([['1.5', '2.7', '3.2'],
                 ['4.1', '5.3', '6.0'],
                 ['7.2', 'invalid', '9.9']], dtype=object)

# Function to convert strings to floats or NaN
def try_convert_to_float(s):
    try:
        float_s = float(s)
        #return np.float64(float_s)
        return float_s
    except ValueError:
        print('failed conversion')
        return s
    
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        print(data[i, j])
        converted = try_convert_to_float(data[i, j])
        print(type(converted))
        data[i, j] = converted
        print(type(data[i, j]))

print(data)

1.5
<class 'float'>
<class 'float'>
2.7
<class 'float'>
<class 'float'>
3.2
<class 'float'>
<class 'float'>
4.1
<class 'float'>
<class 'float'>
5.3
<class 'float'>
<class 'float'>
6.0
<class 'float'>
<class 'float'>
7.2
<class 'float'>
<class 'float'>
invalid
failed conversion
<class 'str'>
<class 'str'>
9.9
<class 'float'>
<class 'float'>
[[1.5 2.7 3.2]
 [4.1 5.3 6.0]
 [7.2 'invalid' 9.9]]


In [47]:
from ProcessOptimizer import Optimizer

opt = Optimizer([(-2.0, 2.0), (0,21), ('foo', 'bar')], "ET", acq_optimizer="sampling")

next2 = opt.ask(2)

print(next2)


[[np.float64(0.20000000000000018), np.int64(9), np.str_('foo')], [np.float64(0.6000000000000001), np.int64(18), np.str_('bar')]]


In [ ]:
space = Space([Real(20, 100, name='x1'), Real(0, 1, name='x2'),
                  Categorical(['A', 'B'], name='x3')])



In [10]:
def sample_space():
    return Space([Real(0, 10, name='x1'), Real(-5, 5, name='x2')])

def test_get_optimal_DOE_without_categorical():
    space = sample_space()
    result, factor_names = get_optimal_DOE(space, 12, design_type='optimization', res=7)
    print(result)
    assert result.shape == (12, 2)
    assert np.all(result[:, 0] >= 0) and np.all(result[:, 0] <= 10)
    assert np.all(result[:, 1] >= -5) and np.all(result[:, 1] <= 5)
    assert factor_names == ['x1', 'x2']

test_get_optimal_DOE_without_categorical()

[[ 0.          5.        ]
 [ 0.         -5.        ]
 [10.          5.        ]
 [ 0.         -1.66666667]
 [ 6.66666667  3.33333333]
 [10.          1.66666667]
 [10.         -5.        ]
 [ 6.66666667 -5.        ]
 [ 1.66666667  1.66666667]
 [ 3.33333333  5.        ]
 [ 3.33333333 -3.33333333]
 [ 8.33333333 -1.66666667]]


In [ ]:
def optimal_design_space():
    return Space([Real(20, 100, name='x1'), Real(0, 1, name='x2'),
                  Categorical(['A', 'B'], name='x3')])

def test_get_optimal_DOE():
    space = optimal_design_space()

    for design_type in ['linear', 'screening', 'response', 'optimization']:

        design, factor_names = get_optimal_DOE(space, 16, design_type=design_type, res=5)

        assert design.shape == (16, 3)
        assert np.all(design[:, 0] >= 20) and np.all(design[:, 0] <= 100)
        assert np.all(design[:, 1] >= 0) and np.all(design[:, 1] <= 1)
        assert [entry in ['A', 'B'] for entry in design[:, 2]]
        assert factor_names == ['x1', 'x2', 'x3']

test_get_optimal_DOE()


def test_custom_model():
    space = optimal_design_space()

    custom_model = "x1 + x2 + x3 + x1:x2 + pow(x1, 2)"

    design, factor_names = get_optimal_DOE(space, 6, res=5, model=custom_model)

    assert design.shape == (6, 3)
    assert np.all(design[:, 0] >= 20) and np.all(design[:, 0] <= 100)
    assert np.all(design[:, 1] >= 0) and np.all(design[:, 1] <= 1)
    assert [entry in ["A", "B"] for entry in design[:, 2]]
    assert factor_names == ["x1", "x2", "x3"]

test_custom_model()

[[np.float64(100.0) np.float64(0.0) 'B']
 [np.float64(60.0) np.float64(0.0) 'A']
 [np.float64(100.0) np.float64(1.0) 'A']
 [np.float64(20.0) np.float64(1.0) 'A']
 [np.float64(20.0) np.float64(0.0) 'B']
 [np.float64(60.0) np.float64(1.0) 'B']]


In [7]:
print(factor_space)

Space([Integer(low=10, high=40),
       Categorical(categories=('A', 'B'), prior=None),
       Categorical(categories=('A', 'B', 'c', 'd'), prior=None),
       Categorical(categories=('A', 'B', 'c', 'd'), prior=None)])


In [ ]:
def sample_space():
    return Space([Real(0, 10, name='x1'), Real(-5, 5, name='x2')])

def test_doe_to_real_space_basic(sample_space):

    design = np.array([[0, 0], [1, 1]])
    result = doe_to_real_space(design, sample_space)
    assert np.asarray(result).shape == (2, 2)
    assert np.allclose(result[0], [0, -5])
    assert np.allclose(result[1], [10, 5])

test_doe_to_real_space_basic(sample_space())

def test_doe_to_real_space_scaler(sample_space):
    design = np.array([[0.25, 0.75], [0.75, 0.25]])
    corner_points = [[0, 0], [1, 1]]
    result = doe_to_real_space(design, sample_space, corner_points=corner_points)
    assert np.allclose(result[0], [2.5, 2.5])
    assert np.allclose(result[1], [7.5, -2.5])

test_doe_to_real_space_scaler(sample_space())



In [6]:
factor_names = ['fw', 'bw', 'x1', 'x2', 'x3']

corner_neg = [-1] * len(factor_names)
corner_pos = [1] * len(factor_names)

print(corner_neg)
print(corner_pos)

[-1, -1, -1, -1, -1]
[1, 1, 1, 1, 1]


In [3]:
from ProcessOptimizer.doe.optimal_design import build_optimal_design

def test_build_optimal_design_vanilla():
    factor_names = ['x1', 'x2', 'x3']

    result = build_optimal_design(factor_names, run_count=12)

    assert result.shape == (12, 3)
    print(result)
    assert np.all(result[:, :2] >= 0) and np.all(result[:, :2] <= 1)
    assert np.all(np.isin(result[:, 2], [-1, 1]))

test_build_optimal_design_vanilla()

[[-1.          1.         -1.        ]
 [-1.          1.          1.        ]
 [-1.         -1.         -1.        ]
 [ 1.          0.          1.        ]
 [-1.         -1.          1.        ]
 [-1.          0.          0.        ]
 [ 0.         -1.          0.        ]
 [ 1.          1.         -1.        ]
 [ 0.          0.03058772 -1.        ]
 [ 1.         -1.         -1.        ]
 [ 1.         -1.          1.        ]
 [ 1.          1.          1.        ]]


AssertionError: 

In [7]:
np.where(0.3 < 0, -1, 1)

array(1)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0., 1.))



design = np.array([[0.25, 0.75], [0.75, 0.25]])

In [3]:
import sys

print(sys.path)

['c:\\Users\\RUCT\\OneDrive - Novo Nordisk\\Python Scripts\\ProcessOptimizer\\ProcessOptimizer\\doe\\various_dev_tests', 'c:\\Users\\RUCT\\AppData\\Local\\anaconda3\\python311.zip', 'c:\\Users\\RUCT\\AppData\\Local\\anaconda3\\DLLs', 'c:\\Users\\RUCT\\AppData\\Local\\anaconda3\\Lib', 'c:\\Users\\RUCT\\AppData\\Local\\anaconda3', '', 'c:\\Users\\RUCT\\AppData\\Local\\anaconda3\\Lib\\site-packages', 'c:\\users\\ruct\\onedrive - novo nordisk\\python scripts\\my_process_optimizer\\processoptimizer', 'c:\\Users\\RUCT\\AppData\\Local\\anaconda3\\Lib\\site-packages\\win32', 'c:\\Users\\RUCT\\AppData\\Local\\anaconda3\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\RUCT\\AppData\\Local\\anaconda3\\Lib\\site-packages\\Pythonwin', 'c:\\Users\\RUCT\\AppData\\Local\\anaconda3\\Lib\\site-packages\\setuptools\\_vendor']


In [ ]:
from ProcessOptimizer.doe.optimal_design import bootstrap, make_model

In [11]:
space = None

In [22]:
from ProcessOptimizer.doe.optimal_design import optimize_design
import patsy

def test_optimize_design():
    factor_names = ['x1', 'x2', 'x3']
    design = np.array([[0.5, 0.5, -1], [0.5, 0.5, 1], [-0.45, 0.45, 0.45]])
    model = "x1 + x2 + x3"
    X = patsy.dmatrix(model, {"x1": [0.5, 0.5, -0.45], "x2": [0.5, 0.5, 0.45], "x3": [-1, 1, 0.45]})
    
    code = "patsy.dmatrix('x1 + x2 + x3', {'x1': [design_point['x1']], 'x2': [design_point['x2']], 'x3': [design_point['x3']]})[0]"
    
    result = optimize_design(X, design, factor_names, code)
    
    assert result.shape == design.shape
    assert np.all(result >= -1) and np.all(result <=  1)

out = test_optimize_design()

NameError: name 'patsy' is not defined